# cellmap-flow on Colab

Run cellmap-flow's inference server in Colab and serve it via a public
Cloudflare Tunnel URL. Free Colab gives you a T4 GPU — ~10× faster than the
CPU-only HF Space free tier. The URL only works while this Colab session is
alive.

**Steps:**
1. Pick a runtime with GPU (Runtime → Change runtime type → T4 GPU).
2. Run all cells.
3. Copy the printed `https://*.trycloudflare.com` URL into the cellmap-flow
   browser dashboard's "Inference server URL" field.
4. Pan around in Neuroglancer; chunks are computed here on the Colab GPU.

## 1. Install cellmap-flow + cloudflared

Takes ~3–5 minutes the first time. cloudflared is a tiny static binary; no
signup or authtoken needed.

In [ ]:
%pip install -q cellmap-flow huggingface_hub s3fs
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

## 2. Configure model + dataset

Pick a cellmap HF model and a public zarr URL. Defaults below run a mito
affinity model on a Janelia mouse-liver dataset.

In [ ]:
HF_REPO = "cellmap/jrc_mus-livers_16nm_to_8nm_mito"
MODEL_NAME = HF_REPO.split("/")[-1]
DATASET = "s3://janelia-cosem-datasets/jrc_mus-liver/jrc_mus-liver.zarr/recon-1/em/fibsem-uint8/"
PORT = 8765

## 3. Start the cellmap-flow server in the background

We run the server as a subprocess so we can also start cloudflared in the
next cell while the server stays alive.

In [ ]:
import subprocess, time

server = subprocess.Popen(
    [
        "cellmap_flow_server", "huggingface",
        "--repo", HF_REPO,
        "--name", MODEL_NAME,
        "-d", DATASET,
        "--port", str(PORT),
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
print(f"server pid={server.pid}, waiting for it to listen on :{PORT} ...")
for _ in range(120):
    line = server.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    if "Running on" in line or f":{PORT}" in line:
        print("\n[server] ready.")
        break

## 4. Start the public Cloudflare Tunnel

Prints the public URL when the tunnel is up. Paste that into the cellmap-flow
browser dashboard.

In [ ]:
import subprocess, re, time

tunnel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = None
for _ in range(120):
    line = tunnel.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break

print("\n" + "=" * 70)
if PUBLIC_URL:
    print("Public URL (paste into the browser dashboard):")
    print("   ", PUBLIC_URL)
else:
    print("Did not detect a public URL in cloudflared output. Scroll up to see why.")
print("=" * 70)

## 5. Keep the runtime alive

Run this cell last so Colab doesn't idle-disconnect. Stop it with the ▢
button when you're done; that also closes the tunnel.

In [ ]:
import time, select

def drain(proc, label):
    """Print whatever proc has emitted since the last read, prefixed."""
    while True:
        r, _, _ = select.select([proc.stdout], [], [], 0)
        if not r:
            return
        line = proc.stdout.readline()
        if not line:
            return
        print(f"[{label}] {line}", end="")

try:
    while True:
        drain(server, "server")
        drain(tunnel, "tunnel")
        if server.poll() is not None:
            drain(server, "server")
            print(f"\n[server] exited rc={server.returncode}. logs above.")
            break
        if tunnel.poll() is not None:
            drain(tunnel, "tunnel")
            print(f"\n[tunnel] exited rc={tunnel.returncode}. logs above.")
            break
        time.sleep(2)
finally:
    for p in (server, tunnel):
        try: p.terminate()
        except Exception: pass
